# Enterprise Multi-Agent AI System — Blackwell 6000

- 1 CEO (LLaMA 7B) + 24 Department LoRA adapters, top-5 routing
- Manual training loop (incompatible with HF Trainer's single-loss expectation)
- 100-step checkpoint → HF auto-upload, resume from HF on failure
- Verified streaming datasets with error fallbacks

In [ ]:
!pip install -q torch transformers datasets accelerate tokenizers huggingface_hub bitsandbytes tqdm

In [ ]:
import os, json, shutil, glob, math, random, sys
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple
from tqdm import tqdm

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.cuda.amp import autocast
from torch.utils.data import DataLoader

from transformers import LlamaConfig, LlamaForCausalLM
from transformers import PreTrainedTokenizerFast, DataCollatorForLanguageModeling
from datasets import load_dataset
from datasets import Dataset as HFDataset
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders
from huggingface_hub import HfApi, hf_hub_download, login
import getpass

print("Imports OK")

In [ ]:
HF_TOKEN = getpass.getpass("HF token: ")
login(HF_TOKEN, add_to_git_credential=False)
api = HfApi()
print("Logged in!")

---
## 1. Config

In [ ]:
@dataclass
class Department:
    id: int
    name: str
    short: str
    description: str
    data_sources: List[str]

DEPARTMENTS = [
    Department(1, "Legal Compliance", "legal", "Regulatory compliance, contract law", ["nvidia/Nemotron-Pretraining-Legal-v1"]),
    Department(2, "Accounting Audit", "audit", "Financial audit, internal controls, GAAP", ["infCapital/investopedia_terms_en"]),
    Department(3, "Finance Funding", "finance", "Capital structure, fundraising", ["JanosAudran/financial-reports-sec"]),
    Department(4, "Design Production", "design", "Product design, manufacturing", ["HuggingFaceFW/fineweb-edu"]),
    Department(5, "Logistics Warehouse", "logistics", "Supply chain, inventory, shipping", ["HuggingFaceFW/fineweb-edu"]),
    Department(6, "Sales Customer Service", "sales", "CRM, sales pipeline, customer support", ["HuggingFaceFW/fineweb-edu"]),
    Department(7, "Marketing", "marketing", "Brand strategy, campaigns, market research", ["HuggingFaceFW/fineweb-edu"]),
    Department(8, "IT", "it", "Infrastructure, cybersecurity", ["transformersbook/codeparrot"]),
    Department(9, "Operation Process", "ops", "Business processes, workflow optimization", ["HuggingFaceFW/fineweb-edu"]),
    Department(10, "Tax", "tax", "Tax planning, compliance, filing", ["infCapital/investopedia_terms_en"]),
    Department(11, "Risk Management", "risk", "Risk assessment, mitigation, insurance", ["HuggingFaceFW/fineweb-edu"]),
    Department(12, "Strategic Planning", "strategy", "Corporate strategy, M&A, expansion", ["HuggingFaceFW/fineweb-edu"]),
    Department(13, "Procurement", "procure", "Vendor management, sourcing", ["HuggingFaceFW/fineweb-edu"]),
    Department(14, "Investment", "invest", "Portfolio management, due diligence", ["JanosAudran/financial-reports-sec"]),
    Department(15, "Quality Control", "quality", "Quality assurance, testing, compliance", ["HuggingFaceFW/fineweb-edu"]),
    Department(16, "Corporate Communications", "comms", "PR, internal comms, crisis management", ["HuggingFaceFW/fineweb-edu"]),
    Department(17, "Fixed Asset Management", "assets", "Asset lifecycle, depreciation, capex", ["infCapital/investopedia_terms_en"]),
    Department(18, "Internal Audit", "intaudit", "Internal controls, fraud prevention", ["nvidia/Nemotron-Pretraining-Legal-v1"]),
    Department(19, "Supply Chain Planning", "scm", "Demand forecasting, supplier planning", ["HuggingFaceFW/fineweb-edu"]),
    Department(20, "Data Analytics", "analytics", "Data science, ML models, dashboards", ["transformersbook/codeparrot"]),
    Department(21, "Treasury Management", "treasury", "Cash management, liquidity", ["JanosAudran/financial-reports-sec"]),
    Department(22, "Business Intelligence", "bi", "Competitive intelligence, market data", ["HuggingFaceFW/fineweb-edu"]),
    Department(23, "Contract Administration", "contract", "Contract lifecycle, renewals, compliance", ["nvidia/Nemotron-Pretraining-Legal-v1"]),
    Department(24, "Asset Valuation", "valuation", "Valuation models, appraisal, impairment", ["JanosAudran/financial-reports-sec"]),
]
NUM_DEPARTMENTS = len(DEPARTMENTS)

@dataclass
class TrainingConfig:
    num_gpus: int = 8
    micro_batch_size: int = 2
    grad_accum_steps: int = 8
    max_seq_len: int = 4096
    adapter_rank: int = 64
    top_k_experts: int = 5
    learning_rate: float = 1e-4
    warmup_steps: int = 500
    weight_decay: float = 0.01
    max_steps: int = 30000
    save_every: int = 100
    log_every: int = 10
    num_epochs: int = 10

cfg = TrainingConfig()
print(f"{NUM_DEPARTMENTS} depts, top-{cfg.top_k_experts} routing, {cfg.max_steps} steps")

---
## 2. Load Datasets (verified streaming with fallbacks)

In [ ]:
MAX_TOTAL = 300000
train_texts = []

def load_safe(name, ds_fn, key, limit, max_len=2048):
    count = 0
    try:
        ds = ds_fn()
        for x in ds:
            if count >= limit or len(train_texts) >= MAX_TOTAL:
                break
            if key in x and x[key] and isinstance(x[key], str):
                train_texts.append(x[key][:max_len])
                count += 1
    except Exception as e:
        print(f"  WARN: {name} failed — {e}")
    print(f"  {name}: {count} loaded ({len(train_texts)} total)")
    return count

def load_safe_multi(name, ds_fn, fields, limit, max_len=2048):
    count = 0
    try:
        ds = ds_fn()
        for x in ds:
            if count >= limit or len(train_texts) >= MAX_TOTAL:
                break
            for f in fields:
                if f in x and x[f] and isinstance(x[f], str):
                    train_texts.append(x[f][:max_len])
                    count += 1
                    break
    except Exception as e:
        print(f"  WARN: {name} failed — {e}")
    print(f"  {name}: {count} loaded ({len(train_texts)} total)")
    return count

print("=== Loading enterprise datasets ===")
load_safe("FineWeb-Edu", lambda: load_dataset("HuggingFaceFW/fineweb-edu", "sample-10BT", split="train", streaming=True), "text", 80000)
load_safe("OpenWebMath", lambda: load_dataset("open-web-math/open-web-math", split="train", streaming=True), "text", 40000)
load_safe_multi("CodeParrot", lambda: load_dataset("transformersbook/codeparrot", split="train", streaming=True), ["content", "text"], 30000)
for cfg_name in ["Nemotron-Pretraining-Legal-Case-Law-Summary", "Nemotron-Pretraining-Legal-eCFR", "Nemotron-Pretraining-Legal-CaseHOLD"]:
    load_safe_multi(f"Nemotron-{cfg_name[-20:]}", lambda c=cfg_name: load_dataset("nvidia/Nemotron-Pretraining-Legal-v1", c, split="train", streaming=True), ["text", "input", "content", "question", "answer"], 25000)
load_safe("Investopedia", lambda: load_dataset("infCapital/investopedia_terms_en", split="train", streaming=True), "text", 20000)
for subset in ["small_lite", "large_lite"]:
    try:
        load_safe_multi(f"SEC-{subset}", lambda s=subset: load_dataset("JanosAudran/financial-reports-sec", s, split="train", streaming=True), ["text", "content", "sentence"], 10000)
    except Exception as e:
        print(f"  WARN: SEC-{subset} also failed — {e}")
load_safe("FineWeb", lambda: load_dataset("HuggingFaceFW/fineweb", "sample-10BT", split="train", streaming=True), "text", 30000)

random.seed(42)
random.shuffle(train_texts)
print(f"\n=== TOTAL: {len(train_texts):,} examples ===")

---
## 3. Train Tokenizer

In [ ]:
if not train_texts:
    raise RuntimeError("No training data loaded. Check dataset sources and internet connection.")

tok = Tokenizer(models.BPE(unk_token="<unk>"))
tok.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
tok.decoder = decoders.ByteLevel()
tok.train_from_iterator(train_texts, trainers.BpeTrainer(vocab_size=16384, special_tokens=["<unk>", "<s>", "</s>", "<pad>"], min_frequency=2))
hf_tokenizer = PreTrainedTokenizerFast(tokenizer_object=tok, unk_token="<unk>", bos_token="<s>", eos_token="</s>", pad_token="<pad>")
print(f"Vocab: {hf_tokenizer.vocab_size}")
hf_tokenizer.save_pretrained("./tokenizer")

---
## 4. Prepare Dataset

In [ ]:
MAX_LENGTH = 512
random.seed(42)
random.shuffle(train_texts)

def encode(texts):
    return hf_tokenizer(texts, truncation=True, max_length=MAX_LENGTH, padding=False)["input_ids"]

dataset = HFDataset.from_dict({"text": train_texts})
dataset = dataset.map(lambda x: {"input_ids": encode(x["text"])}, remove_columns=["text"], desc="Tokenizing")
dataset = dataset.filter(lambda x: len(x["input_ids"]) > 10, desc="Filtering")

collator = DataCollatorForLanguageModeling(tokenizer=hf_tokenizer, mlm=False, pad_to_multiple_of=8)
print(f"{len(dataset)} examples total")
print(f"Batches per epoch: ~{len(dataset) // max(cfg.micro_batch_size, 1)}")

---
## 5. Model: Enterprise Multi-Agent System

In [ ]:
class TaskRouter(nn.Module):
    def __init__(self, input_dim=4096, hidden_dim=2048, num_experts=24, top_k=5):
        super().__init__()
        self.top_k = top_k
        self.net = nn.Sequential(nn.Linear(input_dim, hidden_dim), nn.GELU(), nn.Linear(hidden_dim, num_experts))
    def forward(self, x):
        logits = self.net(x.mean(dim=1))
        vals, idx = torch.topk(logits, self.top_k, dim=-1)
        return F.softmax(vals, dim=-1), idx

class LoRAAdapter(nn.Module):
    def __init__(self, hidden, rank):
        super().__init__()
        self.q = nn.Parameter(torch.zeros(hidden, rank))
        self.qb = nn.Parameter(torch.zeros(rank, hidden))
        self.k = nn.Parameter(torch.zeros(hidden, rank))
        self.kb = nn.Parameter(torch.zeros(rank, hidden))
        self.v = nn.Parameter(torch.zeros(hidden, rank))
        self.vb = nn.Parameter(torch.zeros(rank, hidden))
        self.o = nn.Parameter(torch.zeros(hidden, rank))
        self.ob = nn.Parameter(torch.zeros(rank, hidden))
        self.reset()
    def reset(self):
        for p in self.parameters():
            if p.dim() >= 2:
                nn.init.kaiming_uniform_(p, a=math.sqrt(5))
            else:
                nn.init.zeros_(p)
    def forward(self, h, alpha=1.0):
        return ((h @ self.q) @ self.qb + (h @ self.k) @ self.kb + (h @ self.v) @ self.vb + (h @ self.o) @ self.ob) * alpha

class EnterpriseMultiAgentSystem(nn.Module):
    def __init__(self, vocab_size=16384, hidden=4096, num_layers=32, heads=32, rank=64, num_depts=24, top_k=5, max_seq=4096):
        super().__init__()
        self.num_departments = num_depts
        self.top_k = top_k
        llama_cfg = LlamaConfig(
            vocab_size=vocab_size, hidden_size=hidden, intermediate_size=hidden*4,
            num_hidden_layers=num_layers, num_attention_heads=heads,
            max_position_embeddings=max_seq, rope_theta=10000.0,
            tie_word_embeddings=True, torch_dtype=torch.bfloat16
        )
        self.backbone = LlamaForCausalLM(llama_cfg)
        for p in self.backbone.parameters():
            p.requires_grad = False

        self.adapters = nn.ModuleDict({d.short: LoRAAdapter(hidden, rank) for d in DEPARTMENTS})
        self.adapters["ceo"] = LoRAAdapter(hidden, rank * 2)
        self.router = TaskRouter(input_dim=hidden, num_experts=num_depts, top_k=top_k)
        self.dept_heads = nn.ModuleDict({d.short: nn.Linear(hidden, vocab_size, bias=False) for d in DEPARTMENTS})
        self.ceo_head = nn.Linear(hidden, vocab_size, bias=False)

        for h in self.dept_heads.values():
            nn.init.normal_(h.weight, 0, 0.02)
        nn.init.normal_(self.ceo_head.weight, 0, 0.02)

    def forward(self, input_ids, attention_mask=None, labels=None):
        h = self.backbone.model(input_ids, attention_mask=attention_mask).last_hidden_state
        route_w, route_i = self.router(h[:, 0, :])

        dept_out = {}
        for d in DEPARTMENTS:
            dept_out[d.short] = h + self.adapters[d.short](h)
        ceo_h = h + self.adapters["ceo"](h)

        ceo_logits = self.ceo_head(ceo_h)
        dept_logits = {n: self.dept_heads[n](dh) for n, dh in dept_out.items()}

        loss = None
        if labels is not None:
            shift_labels = labels[:, 1:].reshape(-1)
            ceo_loss = F.cross_entropy(ceo_logits[:, :-1, :].reshape(-1, ceo_logits.size(-1)), shift_labels)
            dept_loss = 0.0
            for dn in dept_logits:
                dept_loss = dept_loss + F.cross_entropy(dept_logits[dn][:, :-1, :].reshape(-1, dept_logits[dn].size(-1)), shift_labels)
            dept_loss = dept_loss / len(dept_logits)
            loss = 0.3 * dept_loss + 0.7 * ceo_loss

        return loss, ceo_logits, dept_logits, route_w

    def generate(self, input_ids, **gen_kwargs):
        return self.backbone.generate(input_ids, **gen_kwargs)

    def save_everything(self, path):
        os.makedirs(path, exist_ok=True)
        torch.save(self.adapters.state_dict(), f"{path}/adapters.pt")
        torch.save(self.router.state_dict(), f"{path}/router.pt")
        torch.save(self.ceo_head.state_dict(), f"{path}/ceo_head.pt")
        torch.save({n: h.state_dict() for n, h in self.dept_heads.items()}, f"{path}/dept_heads.pt")
        self.backbone.save_pretrained(f"{path}/backbone")

    def load_everything(self, path):
        self.adapters.load_state_dict(torch.load(f"{path}/adapters.pt", map_location="cpu"))
        self.router.load_state_dict(torch.load(f"{path}/router.pt", map_location="cpu"))
        self.ceo_head.load_state_dict(torch.load(f"{path}/ceo_head.pt", map_location="cpu"))
        dept_heads_sd = torch.load(f"{path}/dept_heads.pt", map_location="cpu")
        for n, h in self.dept_heads.items():
            h.load_state_dict(dept_heads_sd[n])

model = EnterpriseMultiAgentSystem(
    vocab_size=16384, rank=cfg.adapter_rank,
    num_depts=NUM_DEPARTMENTS, top_k=cfg.top_k_experts
)
model = model.to(torch.bfloat16)

n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
n_total = sum(p.numel() for p in model.parameters())
print(f"Params: {n_total/1e9:.2f}B total, {n_trainable/1e6:.2f}M trainable")

---
## 6. Manual Training Loop (100-step checkpoint, HF resume)

HF Trainer is incompatible with the multi-head architecture. Manual loop gives full control over the hierarchical loss.

In [ ]:
MODEL_NAME = "pink-elephant-enterprise-agent"
REPO_ID = f"pinkelephantlimited/{MODEL_NAME}"
api.create_repo(REPO_ID, private=False, repo_type="model", exist_ok=True)
print(f"Repo {REPO_ID} ready")

In [ ]:
def upload_checkpoint(step):
    ckpt_dir = f"./{MODEL_NAME}/checkpoint-{step}"
    if not os.path.exists(ckpt_dir):
        return
    print(f"Uploading checkpoint-{step} to HF...", end=" ", flush=True)
    HfApi().upload_folder(
        folder_path=ckpt_dir,
        repo_id=REPO_ID,
        path_in_repo=f"checkpoints/checkpoint-{step}",
        ignore_patterns=["*.bin", "optimizer.pt"]
    )
    print("done")

# ─── Resume logic ───
resume_from = None
local_ckpts = sorted(glob.glob(f"./{MODEL_NAME}/checkpoint-*"))
if local_ckpts:
    resume_from = local_ckpts[-1]
    print(f"Found local checkpoint: {resume_from}")
else:
    print("No local checkpoints. Checking HF...")
    try:
        files = api.list_repo_files(REPO_ID, repo_type="model")
        ckpt_ids = sorted(set(
            f.split("/")[1] for f in files
            if f.startswith("checkpoints/checkpoint-") and len(f.split("/")) >= 2
        ))
        if ckpt_ids:
            latest = ckpt_ids[-1]
            dst = f"./{MODEL_NAME}/{latest}"
            os.makedirs(dst, exist_ok=True)
            print(f"Downloading {latest} from HF...")
            for fn in ["trainer_state.json", "adapters.pt", "router.pt", "ceo_head.pt", "dept_heads.pt", "optimizer.pt", "scheduler.pt"]:
                try:
                    p = hf_hub_download(repo_id=REPO_ID, filename=f"checkpoints/{latest}/{fn}", repo_type="model")
                    shutil.copy2(p, os.path.join(dst, fn))
                    print(f"  Got {fn}")
                except Exception:
                    pass
            if os.path.exists(os.path.join(dst, "trainer_state.json")):
                resume_from = dst
                print(f"Resume from HF: {resume_from}")
            else:
                print("Download incomplete, starting fresh")
    except Exception as e:
        print(f"HF check failed: {e}")

# ─── Parse start step & load weights (no model reassignment — marimo compat) ───
start_step = 0
if resume_from:
    step_num = int(os.path.basename(resume_from).split("-")[1])
    start_step = step_num
    print(f"Loading model from {resume_from} (step {start_step})")
    model.load_everything(resume_from)

print(f"Starting from step {start_step}")

In [ ]:
# ─── Optimizer & scheduler ───
model = model.cuda()

optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=cfg.learning_rate, weight_decay=cfg.weight_decay, betas=(0.9, 0.95)
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg.max_steps)

if resume_from:
    opt_path = f"{resume_from}/optimizer.pt"
    if os.path.exists(opt_path):
        optimizer.load_state_dict(torch.load(opt_path, map_location="cuda"))
        print("Restored optimizer")
    sched_path = f"{resume_from}/scheduler.pt"
    if os.path.exists(sched_path):
        scheduler.load_state_dict(torch.load(sched_path, map_location="cpu"))
        print("Restored scheduler")

print(f"Optimizer ready — {len([p for p in model.parameters() if p.requires_grad])} trainable param groups")
print(f"Device: {next(model.parameters()).device}")

In [ ]:
# ─── Training loop ───
# DataLoader is recreated each epoch to prevent exhaustion (critical for correctness)

device = torch.device("cuda")
model.train()
global_step = start_step
optimizer.zero_grad()

pbar = tqdm(total=cfg.max_steps - start_step, desc="Training", unit="step", initial=start_step)

while global_step < cfg.max_steps:
    loader = DataLoader(
        dataset, batch_size=cfg.micro_batch_size, shuffle=True,
        collate_fn=collator, num_workers=2, pin_memory=True,
        drop_last=True
    )
    n_batches = 0

    for batch in loader:
        input_ids = batch["input_ids"].to(device, non_blocking=True)
        labels = batch["labels"].to(device, non_blocking=True)
        attn = batch.get("attention_mask")
        if attn is not None:
            attn = attn.to(device, non_blocking=True)

        with autocast(dtype=torch.bfloat16):
            loss, ceo_logits, dept_logits, route_w = model(input_ids, attention_mask=attn, labels=labels)

        loss.backward()
        n_batches += 1

        if n_batches % cfg.grad_accum_steps == 0:
            torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
            optimizer.step()
            optimizer.zero_grad()
            scheduler.step()
            global_step += 1

            pbar.update(1)
            cur_lr = scheduler.get_last_lr()[0]
            pbar.set_postfix({"loss": f"{loss.item():.4f}", "lr": f"{cur_lr:.2e}"})

            if global_step % cfg.save_every == 0 and global_step > start_step:
                cur_loss = loss.item()
                ckpt_dir = f"./{MODEL_NAME}/checkpoint-{global_step}"
                model.save_everything(ckpt_dir)
                torch.save(optimizer.state_dict(), f"{ckpt_dir}/optimizer.pt")
                torch.save(scheduler.state_dict(), f"{ckpt_dir}/scheduler.pt")
                with open(f"{ckpt_dir}/trainer_state.json", "w") as f:
                    json.dump({
                        "global_step": global_step,
                        "loss": cur_loss,
                        "epoch": global_step * cfg.micro_batch_size * cfg.grad_accum_steps / max(len(dataset), 1)
                    }, f)
                upload_checkpoint(global_step)

            if global_step >= cfg.max_steps:
                break

    if global_step >= cfg.max_steps:
        break

pbar.close()
print(f"Training complete at step {global_step}")

---
## 7. Upload Final Model

In [ ]:
save_dir = f"/tmp/{MODEL_NAME}"
if os.path.exists(save_dir):
    shutil.rmtree(save_dir)

model.save_everything(save_dir)
hf_tokenizer.save_pretrained(save_dir)

api.upload_folder(folder_path=save_dir, repo_id=REPO_ID, ignore_patterns=["*.bin", "optimizer.pt"])
print(f"Uploaded: https://huggingface.co/{REPO_ID}")

---
## 8. Inference Pipeline

In [ ]:
@dataclass
class AgentResponse:
    department: str
    output: str
    confidence: float
    routing_weight: float

@dataclass
class EnterpriseDecision:
    objective: str
    ceo_plan: str
    department_responses: List[AgentResponse]
    ceo_decision: str
    confidence: float
    refinement_rounds: int

class EnterpriseInferencePipeline:
    def __init__(self, model, tokenizer, max_refine=3, conf_thresh=0.85):
        self.model = model
        self.tokenizer = tokenizer
        self.max_refine = max_refine
        self.conf_thresh = conf_thresh

    def __call__(self, objective, verbose=True):
        ceo_plan = self._gen(f"<|ceo|> Decompose: {objective}\nPlan:")
        if verbose:
            print(f"\n{'='*60}\nCEO PLAN\n{'='*60}\n{ceo_plan}\n")
        depts, weights = self._route(objective)
        if verbose:
            print(f"Routing: {len(depts)} depts")
            for d, w in zip(depts, weights):
                print(f"  {d.name}: {w:.2%}")
        responses = []
        for d, w in zip(depts, weights):
            out = self._gen(f"<|{d.short}|> As {d.name}, analyze:\nObjective: {objective}\nContext: {ceo_plan}\nAnalysis:")
            responses.append(AgentResponse(d.short, out, 0.5, w))
            if verbose:
                print(f"\n  [{d.name}] {out[:200]}...")
        decision_text = "\n".join(f"[{r.department}]: {r.output[:200]}" for r in responses)
        decision = self._gen(f"<|decision|> Synthesize based on departments:\n{decision_text}\nFinal decision:")
        conf = sum(r.confidence for r in responses) / max(len(responses), 1) * 0.7 + 0.3
        if verbose:
            print(f"\n{'='*60}\nDECISION (conf: {conf:.1%})\n{'='*60}\n{decision}\n")
        rounds = 0
        while conf < self.conf_thresh and rounds < self.max_refine:
            rounds += 1
            for r in responses:
                if r.confidence < 0.5:
                    r.output = self._gen(f"<|{r.department}|> Refine analysis for {objective} given {decision}\nRefined:")
            decision_text = "\n".join(f"[{r.department}]: {r.output[:200]}" for r in responses)
            decision = self._gen(f"<|decision|> Final synthesis:\n{decision_text}\nFinal:")
            conf = sum(r.confidence for r in responses) / max(len(responses), 1) * 0.7 + 0.3
        return EnterpriseDecision(objective, ceo_plan, responses, decision, conf, rounds)

    def _route(self, objective):
        tokens = self.tokenizer(objective, return_tensors="pt")["input_ids"].cuda()
        with torch.no_grad(), autocast(dtype=torch.bfloat16):
            h = self.model.backbone.model(tokens).last_hidden_state
            w, i = self.model.router(h[:, 0, :])
        return [DEPARTMENTS[idx] for idx in i[0].tolist()], w[0].tolist()

    def _gen(self, prompt, max_tokens=256):
        tokens = self.tokenizer(prompt, return_tensors="pt")["input_ids"].cuda()
        with torch.no_grad():
            out = self.model.backbone.generate(
                tokens, max_new_tokens=max_tokens, do_sample=True,
                temperature=0.7, top_p=0.9,
                pad_token_id=self.tokenizer.pad_token_id
            )
        return self.tokenizer.decode(out[0], skip_special_tokens=True)

print("Inference pipeline defined")

In [ ]:
print("=" * 60)
print("Enterprise Multi-Agent AI System — Blackwell 6000")
print("=" * 60)
print(f"{NUM_DEPARTMENTS} departments, 1 CEO, top-{cfg.top_k_experts} routing")
print(f"Trainable: {n_trainable/1e6:.2f}M / {n_total/1e9:.2f}B")
print(f"Checkpoints: every {cfg.save_every} steps → HF")